In [1]:
import sys
import os
from pathlib import Path
__file__ = os.getcwd()
ROOT_DIR = Path(__file__).parent.parent.parent
print(ROOT_DIR)

sys.path.insert(0, str(ROOT_DIR))

/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT


In [2]:
from videodeepsearch.core.app_state import (
    Appstate,
    get_app_state,
    get_context_file_sys,
    get_external_client,
    get_llm_instance,
    get_postgres_client,
    get_segment_milvus,
    get_storage_client,
    get_visual_image_client,
)

from videodeepsearch.tools.clients import *


app_state = Appstate()

In [3]:
# CONSTANT
MILVUS_HOST = "localhost"
MILVUS_PORT = 19530
MILVUS_URI = f"http://{MILVUS_HOST}:{MILVUS_PORT}"

IMAGE_COLLECTION_NAME = "image_milvus"
IMAGE_VISUAL_PARAM = {
    "type_config": "dense",
    "dimension": 512,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Image embeddings for video frames",
    "index_params": {"M": 64, "efConstruction": 100},
}
IMAGE_CAPTION_DENSE_PARAM = {
    "type_config": "dense",
    "dimension": 768,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Text embeddings for image captions",
    "index_params": {"M": 64, "efConstruction": 100},
}
IMAGE_CAPTION_SPARSE_PARAM = {
    "type_config": "sparse",
    "dimension": 1000000,  # ignored at runtime
    "metric_type": "BM25",
    "index_type": "SPARSE_INVERTED_INDEX",
    "description": "BM25 sparse index for image captions",
    "index_params": {"inverted_index_algo": "DAAT_MAXSCORE"},
}
IMAGE_VISUAL_FIELD = "visual_embedding_field"
IMAGE_CAPTION_FIELD = "caption_embedding_field"
IMAGE_SPARSE_FIELD = "caption_sparse_embedding_field"


SEGMENT_CAPTION_COLLECTION_NAME = "segment_milvus"
SEGMENT_CAPTION_DENSE_PARAM = {
    "type_config": "dense",
    "dimension": 768,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Text embeddings for segment captions",
    "index_params": {"M": 64, "efConstruction": 100},
}
SEGMENT_CAPTION_SPARSE_PARAM = {
    "type_config": "sparse",
    "dimension": 1000000,  # ignored at runtime
    "metric_type": "BM25",
    "index_type": "SPARSE_INVERTED_INDEX",
    "description": "BM25 sparse index for segment captions",
    "index_params": {"inverted_index_algo": "DAAT_MAXSCORE"},
}

SEGMENT_DENSE_FIELD = "caption_embedding_field"
SEGMENT_SPARSE_FIELD = "caption_sparse_embedding_field"



In [4]:
image_milvus_client = ImageMilvusClient(
    uri=MILVUS_URI,
    collection_name=IMAGE_COLLECTION_NAME,
    visual_param=IMAGE_VISUAL_PARAM,
    caption_param=IMAGE_CAPTION_DENSE_PARAM,
    sparse_param=IMAGE_CAPTION_SPARSE_PARAM,
    visual_ann_field=IMAGE_VISUAL_FIELD,
    caption_ann_field=IMAGE_CAPTION_FIELD,
    sparse_field=IMAGE_SPARSE_FIELD,
)

segment_caption_client = SegmentCaptionImageMilvusClient(
    uri=MILVUS_URI,
    collection_name=SEGMENT_CAPTION_COLLECTION_NAME,
    dense_param=SEGMENT_CAPTION_DENSE_PARAM,
    sparse_param=SEGMENT_CAPTION_SPARSE_PARAM,
    dense_field=SEGMENT_DENSE_FIELD,
    sparse_field=SEGMENT_SPARSE_FIELD,
)

await image_milvus_client.connect()
await segment_caption_client.connect()


app_state.image_milvus_client = image_milvus_client
app_state.segment_milvus_client = segment_caption_client

In [5]:
image_emebedding_settings = ImageEmbeddingSettings(
    model_name='open_clip',
    device='cuda',
    batch_size=32
)
text_emebedding_settings = TextEmbeddingSettings(
    model_name='mmbert',
    device='cuda',
    batch_size=32
)

img_txt_client = ImageEmbeddingClient(base_url='http://localhost:8003')
txt_client = TextEmbeddingClient(base_url='http://localhost:8005')


external_client = ExternalEncodeClient(img_text_client=img_txt_client, img_text_settings=image_emebedding_settings, txt_settings=text_emebedding_settings, txt_client=txt_client)

await external_client.connect()
app_state.external_client = external_client 

2025-12-06 13:30:51.271 | INFO     | videodeepsearch.tools.clients.external.base:load_model:126 - service-image-embedding_model_loaded
2025-12-06 13:30:51.275 | INFO     | videodeepsearch.tools.clients.external.base:load_model:126 - service-text-embedding_model_loaded


In [6]:
MINIO_HOST = "localhost"       # use localhost when running locally
MINIO_PORT = 9000
MINIO_USER = "minioadmin"
MINIO_PASSWORD = "minioadmin"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
MINIO_SECURE = False  
MINIO_ENDPOINT = f"{MINIO_HOST}:{MINIO_PORT}"

storage_client = StorageClient(
    host=MINIO_HOST,
    port=MINIO_PORT,#type:ignore
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=MINIO_SECURE,
)

storage_client._ensure_bucket('videotests')
app_state.minio_client = storage_client 

In [7]:

from videodeepsearch.core.config.llm_config import (
    llm_configs
)
from llama_index.llms.google_genai import GoogleGenAI
from dotenv import load_dotenv
import os
print(os.getcwd())
load_dotenv(dotenv_path='/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT/videodeepsearch/.env')

for llm_config in llm_configs:
    agent_name = llm_config.agent_name
    app_state.llm_instance[agent_name] = GoogleGenAI(
        model=llm_config.model_name,
        generation_config=llm_config.generation_config
    )



/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT/videodeepsearch/test/test_notebook2


In [8]:
import json
from datetime import datetime
from typing import Any

from rich.console import Console, Group
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich import box
from rich.syntax import Syntax
from rich.rule import Rule
from rich.padding import Padding
from rich.style import Style
from rich.theme import Theme

# Llama Index Imports
from llama_index.core.workflow import Event, StopEvent
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentStream,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStreamStructuredOutput
)

# 🎨 BRIGHT NEON THEME FOR DARK TERMINALS
custom_theme = Theme({
    "timestamp": "grey70",
    "agent.name": "bold bright_cyan",
    "agent.label": "bold white on deep_sky_blue1", 
    "tool.name": "bold hot_pink",
    "tool.label": "bold white on deep_pink2",
    "success": "bold spring_green1",
    "warning": "bold gold1",
    "error": "bold red1",
    "header": "bold white",
})

class EventHandler:
    def __init__(self, console: Console = None):
        self.console = console or Console(theme=custom_theme)
        self.last_agent = None
        self.stream_active = False

    def _get_time(self) -> str:
        return datetime.now().strftime("%H:%M:%S")

    def _handle_agent_input(self, data: AgentInput) -> None:
        agent_name = data.current_agent_name
        input_msgs = data.input
        self.last_agent = agent_name

        msg_preview = Text()
        for msg in input_msgs:
            role_style = "bold yellow" if msg.role == "user" else "bold magenta"
            msg_preview.append(f"{msg.role.upper()}: ", style=role_style)
            msg_preview.append(f"{str(msg.content)[:200]}", style="bright_white")
            if len(str(msg.content)) > 200:
                msg_preview.append("...", style="grey50")
            msg_preview.append("\n")

        # Bright Blue Border
        self.console.print(Padding(Panel(
            msg_preview,
            title=f"[agent.label] 🤖 {agent_name} [/agent.label] [grey70]received input[/grey70]",
            border_style="bold dodger_blue1",
            box=box.ROUNDED,
            padding=(0, 2)
        ), (1, 0)))

    def _handle_agent_stream(self, data: AgentStream) -> None:
        if not self.stream_active:
            self.console.print()
            self.console.print(f"[agent.name]⚡ {data.current_agent_name}[/agent.name] [grey70]is working...[/grey70]")
            self.stream_active = True

        delta = data.delta
        thinking_delta = data.thinking_delta

        # Thinking: Lighter purple/grey (readable but distinct)
        if thinking_delta:
            self.console.print(thinking_delta, end='', style="italic plum2")
        # Output: Bright Neon Cyan
        elif delta:
            self.console.print(delta, end='', style="bold cyan1")
        # Heartbeat
        else:
            self.console.print(".", end="", style="dim grey50")

    def _handle_agent_output(self, data: AgentOutput) -> None:
        if self.stream_active:
            self.console.print("\n")
            self.stream_active = False

        agent_name = data.current_agent_name
        response = data.response
        tool_calls = data.tool_calls

        content_group = []
        
        if response.content:
            content_group.append(Text(response.content, style="bright_white"))
        
        if tool_calls:
            t_text = Text("\n🛠️  Triggered Tools:", style="bold gold1")
            for tc in tool_calls:
                t_text.append(f"\n  • {tc.tool_name}", style="khaki1")
            content_group.append(t_text)

        # Bright Green Border
        self.console.print(Padding(Panel(
            Group(*content_group),
            title=f"[agent.label] 💬 {agent_name} [/agent.label] [success]Final Answer[/success]",
            border_style="bold spring_green3",
            box=box.HEAVY_EDGE,
            padding=(1, 2)
        ), (1, 0)))

    def _handle_tool_call(self, data: ToolCall) -> None:
        if self.stream_active:
            self.console.print("\n")
            self.stream_active = False

        tool_name = data.tool_name
        tool_id = data.tool_id
        tool_kwargs = data.tool_kwargs

        table = Table(box=box.SIMPLE, show_header=True, header_style="bold gold1", expand=True)
        table.add_column("Argument", style="bold cyan", ratio=1)
        table.add_column("Value", style="bright_white", ratio=3)

        if tool_kwargs:
            for key, value in tool_kwargs.items():
                table.add_row(key, str(value))
        else:
            table.add_row("-", "[grey50]No parameters[/grey50]")

        # Bright Gold Border
        self.console.print(Padding(Panel(
            table,
            title=f"[tool.label] 🔧 Calling: {tool_name} [/tool.label] [grey70]({tool_id})[/grey70]",
            border_style="bold gold1",
            box=box.ROUNDED,
            padding=(0, 1)
        ), (0, 0, 1, 0)))

    def _handle_tool_call_result(self, data: ToolCallResult):
        tool_name = data.tool_name
        tool_output = data.tool_output
        is_error = tool_output.is_error

        # High Contrast Red vs Bright Green
        style = "bold red1" if is_error else "bold chartreuse3"
        border_style = "red1" if is_error else "chartreuse3"
        icon = "❌" if is_error else "✅"
        
        content_str = str(tool_output)
        try:
            parsed = json.loads(content_str)
            # Monokai is good, but 'ansi_dark' sometimes pops more on black.
            # Keeping monokai but ensuring the surrounding text is bright.
            display_content = Syntax(json.dumps(parsed, indent=2), "json", theme="monokai", word_wrap=True)
        except:
            display_content = Text(content_str, style="bright_white")

        self.console.print(Padding(Panel(
            display_content,
            title=f"[{style}]{icon} Tool Output: {tool_name}[/{style}]",
            border_style=border_style,
            box=box.ROUNDED,
            padding=(1, 2)
        ), (0, 0, 1, 0)))

    def _handle_agent_stream_structured_output(self, data: AgentStreamStructuredOutput) -> None:
        output = data.output
        json_str = json.dumps(output, indent=2)
        syntax = Syntax(json_str, "json", theme="monokai", background_color="default")
        
        # Bright Magenta Border
        self.console.print(Padding(Panel(
            syntax,
            title="[bold magenta1]📊 Structured Output Update[/bold magenta1]",
            border_style="bold magenta1",
            box=box.DOUBLE,
        ), (0, 0)))

    def _handle_stop_event(self, data: StopEvent) -> None:
        if self.stream_active:
            self.console.print("\n")
            self.stream_active = False

        result = data.result
        
        if isinstance(result, (dict, list)):
            content = Syntax(json.dumps(result, indent=2), "json", theme="monokai", line_numbers=True)
        else:
            content = Text(str(result), style="bold bright_white")

        # Neon Green / Success finish
        self.console.print(Padding(Panel(
            content,
            title="[bold white on spring_green3] 🏁 Workflow Completed [/bold white on spring_green3]",
            subtitle=f"[grey70]Finished at {self._get_time()}[/grey70]",
            border_style="bold spring_green3",
            box=box.DOUBLE_EDGE,
            padding=(2, 4)
        ), (2, 0)))

    def handle_event(self, event_data: Event):
        event_type = event_data.__class__.__name__

        handler_map = {
            AgentInput.__name__: self._handle_agent_input,
            AgentOutput.__name__: self._handle_agent_output,
            AgentStream.__name__: self._handle_agent_stream,
            ToolCall.__name__: self._handle_tool_call,
            ToolCallResult.__name__: self._handle_tool_call_result,
            AgentStreamStructuredOutput.__name__: self._handle_agent_stream_structured_output,
            StopEvent.__name__: self._handle_stop_event
        }

        if event_type in handler_map:
            handler_map[event_type](event_data)

In [9]:
from sqlalchemy import text

POSTGRE_USER = "prefect"
POSTGRE_PASSWORD = "prefect"
POSTGRE_HOST = "localhost"       # use localhost for local setup
POSTGRE_PORT = 5432
POSTGRE_DB = "prefect"

POSTGRE_DATABASE_URL = (
    f"postgresql+asyncpg://{POSTGRE_USER}:{POSTGRE_PASSWORD}"
    f"@{POSTGRE_HOST}:{POSTGRE_PORT}/{POSTGRE_DB}"
)
postgres_client = PostgresClient(
    database_url=POSTGRE_DATABASE_URL
)
async with postgres_client.get_session() as session:
    result = await session.execute(text("SELECT version();"))
    version = result.scalar_one()
    print(f"🗄️ PostgreSQL connection successful.\n   Version: {version}")

app_state.postgres_client = postgres_client

🗄️ PostgreSQL connection successful.
   Version: PostgreSQL 14.20 (Debian 14.20-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [10]:
from videodeepsearch.agent.definition import  WORKER_AGENT, ORCHESTRATOR_AGENT, PLANNER_AGENT, GREETER_AGENT
from videodeepsearch.agent.base import get_global_agent_registry
from videodeepsearch.tools.implementation.llm.tool import enhance_textual_query, enhance_visual_query
from videodeepsearch.tools.implementation.persist.orc_tool import update_video_context_orc_agent, orc_synthesize_final_findings
from videodeepsearch.tools.implementation.persist.worker_tool import worker_persist_evidence
from videodeepsearch.tools.implementation.scan.tool import get_segments, get_image, extract_frames_by_time_window, extract_frame_time
from videodeepsearch.tools.implementation.search.tool import get_images_from_caption_query, get_images_from_multimodal_query, get_images_from_visual_query, get_segments_from_event_query
from videodeepsearch.tools.implementation.util.tool import get_related_asr_from_image, get_related_asr_from_video_id
from videodeepsearch.tools.implementation.view.worker_tool import worker_view_all_data_handle, worker_view_my_evidence, worker_view_results, worker_view_statistics

from videodeepsearch.agent.agent_as_tool import run_planning_agent_as_tool


In [11]:
# from llama_index.core.workflow.handler import WorkflowHandler #type:ignore
# from typing import cast

# agent_registry = get_global_agent_registry()
# agent_instance = agent_registry.spawn(
#     name=PLANNER_AGENT,
#     llm=get_llm_instance(PLANNER_AGENT)
# )
# user_msg = f"""
# Here is the orchestrator demand: I want to find picture of a dog. 
# """

# handler: WorkflowHandler = agent_instance.run(user_msg=user_msg)
# async for event in handler.stream_events():
#     print(event)
        
# agent_output: AgentOutput = await handler

# response = cast(str, agent_output.response.content)



In [12]:
app_state.llm_instance

{'GREETER_AGENT': GoogleGenAI(callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x74a3888fe5d0>, system_prompt=None, messages_to_prompt=<function messages_to_prompt at 0x74a40807a200>, completion_to_prompt=<function default_completion_to_prompt at 0x74a3f710c040>, output_parser=None, pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'>, query_wrapper_prompt=None, model='gemini-2.5-flash-lite', temperature=0.1, context_window=None, max_retries=3, is_function_calling_model=True, cached_content=None, built_in_tool=None, use_file_api=True),
 'PLANNER_AGENT': GoogleGenAI(callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x74a3886e2990>, system_prompt=None, messages_to_prompt=<function messages_to_prompt at 0x74a40807a200>, completion_to_prompt=<function default_completion_to_prompt at 0x74a3f710c040>, output_parser=None, pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'>, query_wrapper_prompt=None, model='gemini-2.5-p

In [13]:
from videodeepsearch.tools.base.registry import tool_registry
partial_params = {
    'user_id': 'agenttest',
    'list_video_id':['692ad412086ada3a309334ff', '692ad412086ada3a30933500', '692ad412086ada3a30933501', '692ad412086ada3a30933502'], 
}
tools = tool_registry.get_raw_agent_function(
    agent_name=WORKER_AGENT,
    **partial_params
)

In [14]:
from pprint import pprint
pprint(tools)

[<function enhance_visual_query at 0x74a360109b20>,
 <function enhance_textual_query at 0x74a36010a200>,
 <function worker_persist_evidence at 0x74a36010a520>,
 <function get_segments at 0x74a360109da0>,
 <function get_images_from_visual_query at 0x74a36010b060>,
 <function get_images_from_caption_query at 0x74a36010b100>,
 <function get_segments_from_event_query at 0x74a36010b1a0>,
 <function get_images_from_multimodal_query at 0x74a36010b240>,
 <function get_related_asr_from_video_id at 0x74a36010b2e0>,
 <function get_related_asr_from_image at 0x74a36010b380>,
 <function worker_view_all_data_handle at 0x74a36010b420>,
 <function worker_view_results at 0x74a36010b4c0>,
 <function worker_view_my_evidence at 0x74a36010b560>,
 <function worker_view_statistics at 0x74a36010b600>]


In [15]:
raw_query = "I want to view the dog in the park"

variants = [
    "Dog scenarios in many cases",
    "Wide-angle drone sweep capturing the dog from above while it runs along the park walkway",
    "Close-up macro perspective focused on the dog's fur texture as it turns its head",
    "Backlit silhouette shot of the dog against sunset trees with long soft shadows",
    "Low-angle ground-level shot from behind tall grass partially occluding the dog as it moves",
]


evs = tools[0]
result = await evs(
    raw_query=raw_query,
    variants=variants
)

print(result)

A happy dog running freely in a sunny park, vibrant green grass.
Wide-angle drone view of a dog trotting on a park path, surrounded by trees.
Close-up of a dog's textured fur, soft focus background of a park.
Silhouette of a dog at sunset in a park, long shadows, warm light.
Low-angle shot through grass of a dog moving in a park, natural lighting.


In [16]:
raw_query = "I want to view the dog in the park"

variants = [
    "Dog scenarios in many cases",
    "Wide-angle drone sweep capturing the dog from above while it runs along the park walkway",
    "Close-up macro perspective focused on the dog's fur texture as it turns its head",
    "Backlit silhouette shot of the dog against sunset trees with long soft shadows",
    "Low-angle ground-level shot from behind tall grass partially occluding the dog as it moves",
]


evs = tools[1]
result = await evs(
    raw_query=raw_query,
    variants=variants
)

print(result)

Chó chạy dọc lối đi trong công viên, được ghi lại từ trên cao bằng flycam góc rộng.
Ảnh chụp cận cảnh lấy kết cấu lông của chú chó khi nó quay đầu.
Hình ảnh chó với bóng lưng đổ dài dưới ánh sáng ngược của cây cối lúc hoàng hôn.
Từ dưới thấp, qua đám cỏ cao che khuất một phần, có thể thấy chú chó đang di chuyển.
Một chú chó đang chơi đùa trong công viên xanh mướt dưới ánh nắng.
Chú chó với bộ lông vàng óng đang nô đùa trên bãi cỏ rộng.
Cảnh tượng chú chó đang chạy nhảy vui vẻ giữa không gian thiên nhiên.
Ảnh chụp từ xa chú chó đang đi dạo trên con đường mòn trong công viên.
Chú chó với vẻ mặt tinh nghịch đang nhìn thẳng vào ống kính.
Hoàng hôn buông xuống, chú chó nằm nghỉ ngơi dưới bóng cây trong công viên.


In [17]:
# get_images_from_visual_query 

evs = tools[4]
visual_query = "A small child painting in a messy, colorful art studio. Barefoot, crouching low, squeezing a large tube of paint onto a canvas on the floor. Dozens of paint tubes scattered around."
top_k = 50

kwargs = {
    'visual_query': visual_query,
    'top_k': top_k
}
result = await evs(
    **kwargs
)


2025-12-06 13:30:58.834 | INFO     | videodeepsearch.tools.clients.external.base:load_model:126 - service-image-embedding_model_loaded


In [18]:
result

DataHandle(handle_id='86e6c725-46a8-4429-a014-b2b9e11e2280', tool_used=None, summary="=== Image Search Results ===\n    Total images: 50\n    Videos: ['692ad412086ada3a309334ff', '692ad412086ada3a30933501', '692ad412086ada3a30933502']\n    Avg score: 0.623 | Range: 0.591 - 0.666\n    p25: 0.597 | p50: 0.618 | p75: 0.647\n\n\n    Top matches:\n    01. score=0.666 | video=692ad412086ada3a30933502 | time=00:06:28.033 | frame=11641 | s3://agenttest/images/692ad412086ada3a30933502/00011641_00:06:28.033.webp\n02. score=0.666 | video=692ad412086ada3a30933502 | time=00:09:17.600 | frame=16728 | s3://agenttest/images/692ad412086ada3a30933502/00016728_00:09:17.600.webp\n03. score=0.663 | video=692ad412086ada3a30933502 | time=00:09:19.000 | frame=16770 | s3://agenttest/images/692ad412086ada3a30933502/00016770_00:09:19.000.webp\n04. score=0.662 | video=692ad412086ada3a30933502 | time=00:06:26.100 | frame=11583 | s3://agenttest/images/692ad412086ada3a30933502/00011583_00:06:26.100.webp\n05. score=0

In [19]:
"""
<function get_related_asr_from_video_id at 0x74e8a2ff27a0>,
<function get_related_asr_from_image at 0x74e8a2ff28e0>,
"""

evs = tools[5]
visual_query = "Một bé nhỏ đang vẽ trong một phòng nghệ thuật bừa bộn, đầy màu sắc. Bé đi chân trần, khom người thật thấp, bóp một tuýp sơn lớn cho chảy xuống tấm canvas đặt trên sàn. Xung quanh là hàng chục tuýp sơn vương vãi khắp nơi"
top_k = 50

kwargs = {
    'caption_query': visual_query,
    'top_k_each_request': 100,
    'top_k_final': 50,
    'weights': [0.5, 0.5]
}
result = await evs(
    **kwargs
)



2025-12-03 20:00:31.359 | INFO     | videodeepsearch.tools.clients.external.base:load_model:126 - service-text-embedding_model_loaded


In [20]:
result

DataHandle(handle_id='30045d33-9d5a-4c1b-9178-36ecf0f4f52a', tool_used=None, summary="=== Image Search Results ===\n    Total images: 50\n    Videos: ['692ad412086ada3a30933500', '692ad412086ada3a309334ff', '692ad412086ada3a30933501', '692ad412086ada3a30933502']\n    Avg score: 0.823 | Range: 0.492 - 0.987\n    p25: 0.496 | p50: 0.975 | p75: 0.979\n\n\n    Top matches:\n    01. score=0.987 | video=692ad412086ada3a30933502 | time=00:09:18.300 | frame=16749 | s3://agenttest/images/692ad412086ada3a30933502/00016749_00:09:18.300.webp\n02. score=0.985 | video=692ad412086ada3a30933502 | time=00:06:28.033 | frame=11641 | s3://agenttest/images/692ad412086ada3a30933502/00011641_00:06:28.033.webp\n03. score=0.985 | video=692ad412086ada3a30933502 | time=00:09:22.100 | frame=16863 | s3://agenttest/images/692ad412086ada3a30933502/00016863_00:09:22.100.webp\n04. score=0.984 | video=692ad412086ada3a30933502 | time=00:09:20.900 | frame=16827 | s3://agenttest/images/692ad412086ada3a30933502/00016827_00

In [21]:
# get_segments_from_event_query
evs = tools[6]
visual_query = "Một bé nhỏ đang vẽ trong một phòng nghệ thuật bừa bộn, đầy màu sắc. Bé đi chân trần, khom người thật thấp, bóp một tuýp sơn lớn cho chảy xuống tấm canvas đặt trên sàn. Xung quanh là hàng chục tuýp sơn vương vãi khắp nơi"
top_k = 50

kwargs = {
    'event_query': visual_query,
    'top_k_each_request': 100,
    'top_k_final': 50,
    'weights': [0.5, 0.5]
}
result = await evs(
    **kwargs
)

2025-12-03 20:00:34.130 | INFO     | videodeepsearch.tools.clients.external.base:load_model:126 - service-text-embedding_model_loaded


In [22]:
result

DataHandle(handle_id='3680c1fe-c7ca-4841-a3b6-a342c9d4a2b8', tool_used=None, summary="=== Segment Caption Search Results ===\n    Total segments: 50\n    Videos: ['692ad412086ada3a30933500', '692ad412086ada3a309334ff', '692ad412086ada3a30933501', '692ad412086ada3a30933502']\n    Avg score: 0.710 | Range: 0.486 - 0.983\n    p25: 0.487 | p50: 0.495 | p75: 0.969\n\n    Top matches:\n    01. score=0.983 | video=692ad412086ada3a30933502 | 00:09:19.733 → 00:09:24.500 | frames 16792-16935 | caption [Dựa trên các khung hình được cung cấp, sự kiện trong video mô tả một đứa trẻ đang say mê sáng tạo nghệ thuật trong một không gian làm việc đầy màu sắc. **Mô tả chi tiết sự kiện:** Trong các khung hình, một đứa trẻ (dường như là bé ...] | url: s3://agenttest/caption/segment/692ad412086ada3a30933502/16792_16935_00:09:19.733_00:09:24.500.json\n02. score=0.977 | video=692ad412086ada3a30933502 | 00:06:25.800 → 00:06:27.067 | frames 11574-11612 | caption [Dựa trên các khung hình được cung cấp, ta thấy m

In [23]:
result._raw_data

[SegmentInterface(id='f2194809-4f6d-4134-b9c8-303a93520e45', related_video_id='692ad412086ada3a30933502', minio_path='s3://agenttest/caption/segment/692ad412086ada3a30933502/16792_16935_00:09:19.733_00:09:24.500.json', user_bucket='agenttest', start_frame=16792, end_frame=16935, start_time='00:09:19.733', end_time='00:09:24.500', segment_caption='Dựa trên các khung hình được cung cấp, sự kiện trong video mô tả một đứa trẻ đang say mê sáng tạo nghệ thuật trong một không gian làm việc đầy màu sắc.\n\n**Mô tả chi tiết sự kiện:**\n\nTrong các khung hình, một đứa trẻ (dường như là bé gái tóc vàng, còn rất nhỏ tuổi) đang đứng chân trần trên một sàn nhà được phủ đầy các vết sơn màu loang lổ, cho thấy đây là khu vực thường xuyên được dùng để vẽ tranh. Đứa trẻ mặc một chiếc áo dài tay và quần short đều bị dính nhiều vết sơn rực rỡ.\n\nỞ khung hình đầu tiên (ảnh 1), hành động tập trung vào việc sử dụng một con lăn sơn (roller) để tô một mảng màu đỏ tươi lên một tấm toan lớn đặt dưới sàn, trên nề

In [24]:
# get_images_from_multimodal_query
evs = tools[7]
visual_query = 'A small child painting in a messy, colorful art studio. Barefoot, crouching low, squeezing a large tube of paint onto a canvas on the floor. Dozens of paint tubes scattered around.'
caption_query = "Một bé nhỏ đang vẽ trong một phòng nghệ thuật bừa bộn, đầy màu sắc. Bé đi chân trần, khom người thật thấp, bóp một tuýp sơn lớn cho chảy xuống tấm canvas đặt trên sàn. Xung quanh là hàng chục tuýp sơn vương vãi khắp nơi"

top_k = 50

kwargs = {
    'visual_query': visual_query,
    'caption_query': caption_query,
    'top_k_each_request': 100,
    'top_k_final': 50,
    'weights': [0.5, 0.4, 0.1]
}
result = await evs(
    **kwargs
)

2025-12-03 20:00:34.177 | INFO     | videodeepsearch.tools.clients.external.base:load_model:126 - service-image-embedding_model_loaded
2025-12-03 20:00:34.187 | INFO     | videodeepsearch.tools.clients.external.base:load_model:126 - service-text-embedding_model_loaded


In [25]:
result

DataHandle(handle_id='46d46280-57f6-4624-8c62-4cf31f72966c', tool_used=None, summary="=== Image Search Results ===\n    Total images: 50\n    Videos: ['692ad412086ada3a30933500', '692ad412086ada3a309334ff', '692ad412086ada3a30933501', '692ad412086ada3a30933502']\n    Avg score: 0.613 | Range: 0.419 - 0.824\n    p25: 0.488 | p50: 0.489 | p75: 0.790\n\n\n    Top matches:\n    01. score=0.824 | video=692ad412086ada3a30933502 | time=00:06:28.033 | frame=11641 | s3://agenttest/images/692ad412086ada3a30933502/00011641_00:06:28.033.webp\n02. score=0.821 | video=692ad412086ada3a30933502 | time=00:09:18.300 | frame=16749 | s3://agenttest/images/692ad412086ada3a30933502/00016749_00:09:18.300.webp\n03. score=0.821 | video=692ad412086ada3a30933502 | time=00:06:26.100 | frame=11583 | s3://agenttest/images/692ad412086ada3a30933502/00011583_00:06:26.100.webp\n04. score=0.818 | video=692ad412086ada3a30933502 | time=00:09:23.300 | frame=16899 | s3://agenttest/images/692ad412086ada3a30933502/00016899_00

In [26]:
evs = tools[9]

kwargs = {
    'image_related_video_id': '692ad412086ada3a30933502',
    'image_timestamp': '00:09:18.300',
    'window_seconds': 10.0
}
result = await evs(
    **kwargs
)

In [27]:
result

'\n    Here are the asr captured around the image at timestamp/frame_index: 00:09:18.300/16749 around the window seconds: 10.0\n    \n        ---------------------\n        Start time/index: 00:09:13.840/00016616 - End time/index: 00:10:00.240/00018008\n        ASR content: tại bangvarina đông nam nước đức cô bé ba tuổi loren sport sau một lần đi chơi cùng gia đình vào năm ngoái đã khiến phụ huynh sững sờ trước những bức vẽ của mình đến nay các tác phẩm của loren đều có giá vài ngàn euro cho mỗi bức hầu như tác phẩm nào của bé cũng đã có chủ có những bước chúng tôi giữ làm kỷ niệm như bức vẽ đầu tiên của con hoặc bức cần thích nhất chúng tôi muốn mọi việc rõ ràng và đã liên hệ với sở thuế đảm bảo nộp thuế đúng luật và mở một tài khoản riêng để đảm bảo quyền lợi của con về sau lịch trình của cậu bé ba tuổi khá bận rộn ngoài giờ chơi cùng chú voi nhựa yêu thích cầu sẽ dành cả ngày để trộn màu tô vẽ tùy theo sức sáng tạo của mình\n        ---------------------\n        \n    The asr might

In [28]:
evs = tools[8]

kwargs = {
    'segment_related_video_id': '692ad412086ada3a30933502',
    'segment_start_time': '00:09:18.300',
    'segment_end_time': '00:09:24.500',
    'window_seconds': 10.0
}
result = await evs(
    **kwargs
)

In [29]:
print(result)


    ASR transcript context around the segment:

    ▶ Segment range: 00:09:18.300 → 00:09:24.500
    ▶ Frame range: 16749 → 16935
    ▶ Context window: ±10.0 seconds

    --------------------- TRANSCRIPT CONTEXT ---------------------
    Start time/index: 00:09:13.840/00016616
End time/index:   00:10:00.240/00018008
ASR content:      tại bangvarina đông nam nước đức cô bé ba tuổi loren sport sau một lần đi chơi cùng gia đình vào năm ngoái đã khiến phụ huynh sững sờ trước những bức vẽ của mình đến nay các tác phẩm của loren đều có giá vài ngàn euro cho mỗi bức hầu như tác phẩm nào của bé cũng đã có chủ có những bước chúng tôi giữ làm kỷ niệm như bức vẽ đầu tiên của con hoặc bức cần thích nhất chúng tôi muốn mọi việc rõ ràng và đã liên hệ với sở thuế đảm bảo nộp thuế đúng luật và mở một tài khoản riêng để đảm bảo quyền lợi của con về sau lịch trình của cậu bé ba tuổi khá bận rộn ngoài giờ chơi cùng chú voi nhựa yêu thích cầu sẽ dành cả ngày để trộn màu tô vẽ tùy theo sức sáng tạo của mì

In [30]:
evs = tools[3]

kwargs = {
    'pivot_segment_related_video_id': '692ad412086ada3a30933502',
    'pivot_segment_start_frame': 16792,
    'pivot_segment_end_frame': 16935,
    'hop': 10,
    'include_within_range': True,
    'forward_or_backward': 'forward'
}
result = await evs(
    **kwargs
)


In [31]:
print(result)

DataHandle(handle_id=c49cfae4-a42a-4133-b2c4-c7a04e813400, tool_used=None, summary='=== Segment Caption Search Results ===\n    Total segments: 10\n    Videos: [\'692ad412086ada3a30933502\']\n    Avg score: 0.000 | Range: 0.000 - 0.000\n    p25: 0.000 | p50: 0.000 | p75: 0.000\n\n    Top matches:\n    01. score=0.000 | video=692ad412086ada3a30933502 | 00:09:24.533 → 00:09:27.733 | frames 16936-17032 | caption [Trong các khung hình được cung cấp, một em bé tóc vàng với khuôn mặt bầu bĩnh đang say mê hoạt động vẽ tranh hoặc chơi với màu sắc. Em bé mặc một chiếc áo trắng đã bị vấy bẩn rất nhiều bởi các vết sơn màu tím, vàng, xanh ...] | url: s3://agenttest/caption/segment/692ad412086ada3a30933502/16936_17032_00:09:24.533_00:09:27.733.json\n02. score=0.000 | video=692ad412086ada3a30933502 | 00:09:24.533 → 00:09:27.733 | frames 16936-17032 | caption [Trong các khung hình được cung cấp, một em bé tóc vàng với khuôn mặt bầu bĩnh đang say mê hoạt động vẽ tranh hoặc chơi với màu sắc. Em bé mặc 

In [32]:
from llama_index.core.workflow import Context
from llama_index.core.llms import MockLLM
from llama_index.core.agent import FunctionAgent
llm = MockLLM()
agent = FunctionAgent(llm=llm)

In [33]:
from videodeepsearch.agent.context.worker_context import SmallWorkerContext
agent_name = "agent_name"

context = Context(agent)
async with context.store.edit_state() as ctx_state:
    ctx_state[agent_name] = SmallWorkerContext(worker_agent_name='agent_name', task_objective='...').model_dump()

In [34]:
# get_images_from_visual_query 

evs = tools[4]
visual_query = "A small child painting in a messy, colorful art studio. Barefoot, crouching low, squeezing a large tube of paint onto a canvas on the floor. Dozens of paint tubes scattered around."
top_k = 50

kwargs = {
    'visual_query': visual_query,
    'top_k': top_k
}
result = await evs(
    **kwargs
)

2025-12-03 20:00:39.248 | INFO     | videodeepsearch.tools.clients.external.base:load_model:126 - service-image-embedding_model_loaded


In [35]:
from videodeepsearch.agent.context.worker_context import SmallWorkerContext

from uuid import uuid4

async with context.store.edit_state() as ctx_state:
    agent_context = ctx_state[agent_name]
    small_worker_context = SmallWorkerContext.model_validate(agent_context)
    persist_result = small_worker_context.raw_result_store

    tool_call = ToolCall(tool_name='get_images_from_visual_query', tool_kwargs=kwargs, tool_id=str(uuid4()))
    result.tool_used=tool_call
    persist_result.persist_handle(
        data_handle=result
    )
    ctx_state[agent_name] = small_worker_context # persist result    

    


In [36]:
evs = tools[10]
agent_name = "agent_name"

kwargs = {
    'ctx': context,
    'agent_name': agent_name,
}
result = await evs(
    **kwargs
)


In [37]:
result

'Here is the available handle id(s): 22f0ba3b-f59d-4998-8853-0378db24fcfb'

In [39]:
evs = tools[11]
agent_name = "agent_name"

kwargs = {
    'ctx': context,
    'agent_name': agent_name,
    'handle_id': '22f0ba3b-f59d-4998-8853-0378db24fcfb',
    'slicing': [0,2]
}
result = await evs(
    **kwargs
)
pprint(result)

('Handle id: 22f0ba3b-f59d-4998-8853-0378db24fcfb\n'
 'Tool uses patterns: Tool name: get_images_from_visual_query\n'
 "Tool input kwargs: {'visual_query': 'A small child painting in a messy, "
 'colorful art studio. Barefoot, crouching low, squeezing a large tube of '
 "paint onto a canvas on the floor. Dozens of paint tubes scattered around.', "
 "'top_k': 50}\n"
 'Total: 2 images\n'
 'Top 5:\n'
 '   0. \n'
 '        score=0.666 | 692ad412086ada3a30933502 @ Timestamp/FrameIndex: '
 '00:06:28.033/11641 | minio_pat: '
 's3://agenttest/images/692ad412086ada3a30933502/00011641_00:06:28.033.webp | '
 'image caption: Bức ảnh ghi lại một khoảnh khắc đầy màu sắc và năng động '
 'trong một xưởng vẽ hoặc không gian sáng tạo ngập tràn sơn.\n'
 '\n'
 '**Bối cảnh:**\n'
 'Khung cảnh dường như là một phòng làm việc hoặc phòng trưng bày, với sàn nhà '
 'được phủ kín bởi những vết sơn bắn tung tóe đủ màu sắc, tạo nên một bức '
 'tranh trừu tượng khổng lồ dưới chân. Xung quanh có nhiều bức tranh lớn đ

In [40]:
evs = tools[13]
agent_name = "agent_name"

kwargs = {
    'ctx': context,
    'agent_name': agent_name,
    'handle_id': '22f0ba3b-f59d-4998-8853-0378db24fcfb',
    'group_by': 'video_id'
}
result = await evs(
    **kwargs
)
print(result)

=== Statistic Report (Images) ===
Handle ID: 22f0ba3b-f59d-4998-8853-0378db24fcfb
Tool: get_images_from_visual_query | Strategy: video_id
Total Items: 50
----------------------------------------
Group: 692ad412086ada3a309334ff
  • Count: 5
  • Avg Score: 0.595
  • Best Match: score=0.600 | 692ad412086ada3a309334ff @ Timestamp/FrameIndex: 00:12:27.880/18697 | minio_path: s3://agenttest/images/692ad412086ada3a309334ff/00018697_00:12:27.880.webp

Group: 692ad412086ada3a30933501
  • Count: 4
  • Avg Score: 0.597
  • Best Match: score=0.600 | 692ad412086ada3a30933501 @ Timestamp/FrameIndex: 00:04:08.933/7468 | minio_path: s3://agenttest/images/692ad412086ada3a30933501/00007468_00:04:08.933.webp

Group: 692ad412086ada3a30933502
  • Count: 41
  • Avg Score: 0.629
  • Best Match: score=0.666 | 692ad412086ada3a30933502 @ Timestamp/FrameIndex: 00:06:28.033/11641 | minio_path: s3://agenttest/images/692ad412086ada3a30933502/00011641_00:06:28.033.webp



In [41]:
evs = tools[2]
agent_name = "agent_name"

kwargs = {
    'ctx': context,
    'agent_name': agent_name,
    'handle_id': '22f0ba3b-f59d-4998-8853-0378db24fcfb',
    'confidence_score': 7,
    'claims': "dasdad",
    'slicing': [0,2]
}
result = await evs(
    **kwargs
)
print(result)

✓ Persisted 2 items as evidence (confidence: 7/10, videos: ['692ad412086ada3a30933502'])


In [42]:
evs = tools[12]
agent_name = "agent_name"

kwargs = {
    'ctx': context,
    'agent_name': agent_name,
}
result = await evs(
    **kwargs
)
print(result)

Your Evidence Summary (1 items)

1. 


        From tool:tool_name='get_images_from_visual_query' tool_kwargs={'visual_query': 'A small child painting in a messy, colorful art studio. Barefoot, crouching low, squeezing a large tube of paint onto a canvas on the floor. Dozens of paint tubes scattered around.', 'top_k': 50} tool_id='5076d54e-87e5-4ea1-ba3e-82e711604495'
        Artifacts: 
        
        score=0.666 | 692ad412086ada3a30933502 @ Timestamp/FrameIndex: 00:06:28.033/11641 | minio_pat: s3://agenttest/images/692ad412086ada3a30933502/00011641_00:06:28.033.webp | image caption: Bức ảnh ghi lại một khoảnh khắc đầy màu sắc và năng động trong một xưởng vẽ hoặc không gian sáng tạo ngập tràn sơn.

**Bối cảnh:**
Khung cảnh dường như là một phòng làm việc hoặc phòng trưng bày, với sàn nhà được phủ kín bởi những vết sơn bắn tung tóe đủ màu sắc, tạo nên một bức tranh trừu tượng khổng lồ dưới chân. Xung quanh có nhiều bức tranh lớn đang dựa vào tường hoặc sàn nhà, với gam màu chủ đạo là